# 🔬 ISIC 2024 Skin Cancer Detection — Kaggle GPU Training Notebook

This notebook automates end-to-end 5-fold competition training of the **Multimodal FusionModel** (`EfficientNetV2` + `MetadataMLP` + `Ugly Duckling` scores) using Kaggle GPU acceleration (P100 / T4).

### Pipeline Highlights:
- **Zero Code Modifications Required**: Auto-detects Kaggle GPU, dataset paths, and CUDA FP16.
- **5-Fold GroupKFold**: Leakage-free patient-level cross-validation.
- **Rank-Averaged Ensemble**: Blends predictions across 5 fold checkpoints.
- **Submission Generator**: Automatically exports `submission.csv` and compresses checkpoints for download.

In [ ]:
# Step 1: Clone Repository & Setup Environment
import os
from pathlib import Path

REPO_URL = 'https://github.com/AyushBhardwaj132/Skin-cancer-detection-.git'
REPO_DIR = '/kaggle/working/Skin-cancer-detection-'

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -r requirements.txt


In [ ]:
# Step 2: Check GPU Hardware Acceleration
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    !nvidia-smi


In [ ]:
# Step 3: Run Full 5-Fold GroupKFold Competition Training Pipeline
# Note: Uses train_kaggle.py with automatic Kaggle path detection & FP16
!python train_kaggle.py --all-folds --epochs 10


In [ ]:
# Step 4: Run Inference & Generate Competition Submission CSV
!python main.py infer --method rank


In [ ]:
# Step 5: Verify Submission Format & Zip Checkpoints for Download
import pandas as pd
sub_path = Path('/kaggle/working/outputs/predictions/submission.csv')
if sub_path.exists():
    sub_df = pd.read_csv(sub_path)
    print(f'Submission generated successfully! Shape: {sub_df.shape}')
    print(sub_df.head())

# Compress checkpoints for local download
!zip -q -r /kaggle/working/isic2024_checkpoints.zip /kaggle/working/outputs/checkpoints/
print('Checkpoints zipped to /kaggle/working/isic2024_checkpoints.zip')
